Суть следующая: у вас есть набор данных о продажах и продуктах интернет-магазина. Перед вами стоит 15 различных вопросов, ответить на которые можно, проанализировав эти данные. Ваша задача - написать код для решения каждой задачи и вписать в специальную форму получившийся ответ.

In [2]:
import pandas as pd

In [3]:
sales = pd.read_csv('https://gist.githubusercontent.com/andron23/0675d643fe001b08a66271402a7b609b/raw/231deb6c35c43ca9089b197d94d7d48e093bfb06/sales_data.csv')
products = pd.read_csv('https://gist.githubusercontent.com/andron23/fc2b28de07d5017f951c559f51fcb286/raw/be9f8cd81b28f3c19d314ef908aa68e9f41ae975/products_data.csv')

In [4]:
sales.head()

,order_id,product_id,customer_id,order_date,quantity,price_per_unit,total_price,payment_method,region
0,1,101,1001,2022-02-15,2,500,1000,Карта,Север
1,2,102,1002,2022-03-20,1,800,800,Наличные,Юг
2,3,103,1003,2022-03-25,3,300,900,Карта,Запад
3,4,104,1001,2022-04-10,2,400,800,Карта,Восток
4,5,105,1004,2022-04-15,1,700,700,Наличные,Север


In [5]:
products.head()

,product_id,product_name,category,manufacturer
0,101,Ноутбук HP Pavilion 15,Ноутбуки,A
1,102,Смартфон Samsung Galaxy S21,Смартфоны,B
2,103,Планшет Apple iPad Air,Планшеты,C
3,104,Наушники Sony WH-1000XM4,Наушники,D
4,105,Телевизор LG OLED CX,Телевизоры,E


## 1
В каком регионе было продано наибольшее количество продуктов категории "Смартфоны"?

In [6]:
# Совмещаем датафреймы
all_info = sales.merge(products, on='product_id', how='left')

# Отбираем покупки смартфонов и считаем количество покупок по регионам
all_info[all_info['category'] == 'Смартфоны'] \
    .groupby('region') \
    .agg({'quantity' : 'sum'})

,quantity
region,
Восток,4
Запад,2
Север,1
Юг,3


### Значит ответ на первый вопрос - Восток

# 2
Какова разница между максимальной и минимальной средней ценой продуктов категории "Ноутбуки" между всеми производителями? Ответ округлите до целых.

In [7]:
# Фильтруем по категории, группируем по производителям и находим среднее в каждой категории,
# затем находим разность между максимальной и минимальной средней ценой

mean_price = all_info[all_info['category'] == 'Ноутбуки'] \
    .groupby('manufacturer') \
    .agg({'price_per_unit' : 'mean'}) \
    .reset_index()
print(int(max(mean_price['price_per_unit'] - min(mean_price['price_per_unit']))))

200


# 3
Какой покупатель совершил заказы на наибольшую сумму за все время?

In [8]:
# Группируем по покупателям, затем суммируем total_price по каждому покупателю
all_info.groupby('customer_id') \
    .agg({'total_price' : 'sum'}) \
    .idxmax()

total_price    1003
dtype: int64

# 4 
Какова доля продаж продуктов категории "Планшеты" от общей выручки за все время (учитывая только продукты, информация о которых есть в _products_data_)? Ответ округлите до 2 знака после запятой. Разделитель разряда - точка.

In [9]:
# Группируем данные по категории, считаем выручку по каждой категории и находим долю
merged_data = sales.merge(products, on='product_id')
total_revenue = merged_data.total_price.sum()
merged_data.groupby('category').total_price.sum() / total_revenue

category
Наушники      0.179798
Ноутбуки      0.234343
Планшеты      0.248485
Смартфоны     0.143434
Телевизоры    0.193939
Name: total_price, dtype: float64

# 5 
Какова абсолютная разница в средней цене за единицу продуктов категории "Смартфоны" между двумя регионами с наибольшей общей выручкой по всем категориям товаров (учитывая только продукты, информация о которых есть в products_data)? Ответ округлите до целых и возьмите по модулю.

In [10]:
# Определяем 2 региона с наибольшей общей выручкой по всем категориям товаров, затем считаем в каждом из регионов среднюю цену за единицу смартфонов
merged_data = sales.merge(products, on='product_id')
top2 = merged_data.groupby('region') \
    .total_price.sum() \
    .sort_values(ascending=False) \
    .reset_index().head(2).region.to_list()

mean_smartfon_price = merged_data[(merged_data['category'] == 'Смартфоны') & (merged_data['region'].isin(top2))] \
    .groupby('region').price_per_unit.mean()

round(mean_smartfon_price.diff().abs().iloc[-1])

275

# 6
Сколько в среднем уникальных категорий продуктов было продано по всем регионам?

In [11]:
# Группируем по регионам покупки, затем считаем количество уникальных категорий в каждой группе

all_info.groupby('region') \
    .agg({'category' : 'nunique'}).mean()

category    5.0
dtype: float64

# 7
Какой производитель представлен в наибольшем количестве различных категорий продуктов?

In [12]:
# Сгруппируем наши данные по производителям и посчитаем количество уникальных категорий продуктов у каждого производителя
all_info.groupby('manufacturer') \
    .agg({'category' : 'nunique'}) \
    .idxmax()

category    E
dtype: object

# 8
В каком месяце было продано больше всего продуктов категории "Планшеты" в регионе "Север"?

In [13]:
# Фильтруем данные по категории и региону, затем группируем по месяцам и считаем количество проданных планшетов в каждом месяце
merge_sales_prod = sales.merge(products, on='product_id')
filter_msp = merge_sales_prod[(merge_sales_prod['category'] == 'Планшеты') & (merge_sales_prod['region'] == 'Север')]
# Добавим столбец с месяцами
filter_msp['month'] = pd.to_datetime(filter_msp['order_date']).dt.month_name()

filter_msp.groupby('month') \
    .agg({'quantity' : 'sum'}) \
    .idxmax()


C:\Users\79125\AppData\Local\Temp\ipykernel_25868\3994605349.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filter_msp['month'] = pd.to_datetime(filter_msp['order_date']).dt.month_name()


quantity    August
dtype: object

# 9
Какой производитель продал продукты на наибольшую сумму в регионе "Юг" за март 2023 года?

In [21]:
# Фильтруем данные по дате и региону, затем группируем по производителям и считаем суммы

all_info[(all_info['region'] == 'Юг') & (all_info['order_date'].str.contains('2023-03', na=False))] \
    .groupby('manufacturer').total_price.sum()

manufacturer
B    700
Name: total_price, dtype: int64

# 10
Сколько покупателей совершили заказы на продукты хотя бы трех разных категорий?

In [30]:
# Группируем данные по пользователям, считаем количество уникальных категорий у каждого и отбираем тех, у кого больше трех, затем смотрим на размер этого датафрейма

all_info.groupby('customer_id').category.nunique().reset_index().query('category >= 3').shape[0]

1

# 11
Какова доля продаж продуктов производителя "C" от общей выручки за все время в регионе "Север"? Ответ округлите до 2 знака после запятой. В качестве разделителя разрядов используйте точку.

In [33]:
# Считаем общую выручку по всем товарам в регионе Север. Фильтруем наши данные по производителю и региону и считаем выручку. Затем делим и округляем
merged_data = sales.merge(products, on='product_id')
all_revenue = merged_data[merged_data['region'] == 'Север'].total_price.sum()

c_north_revenue = merged_data[(merged_data['manufacturer'] == 'C') & (merged_data['region'] == 'Север')] \
    .total_price.sum()

round(c_north_revenue/all_revenue, 2)

np.float64(0.27)

# 12 
Найдите покупателей, которые купили более чем 15% уникальных связок "Категория - Производитель". Если таких покупателей несколько - напишите их через запятую с пробелом в порядке возрастания.

In [54]:
merged_data = sales.merge(products, on='product_id')

total_man = int(merged_data.groupby(['category', 'manufacturer'])['manufacturer'].agg('nunique').sum())

unique_per_user = merged_data.groupby(['customer_id', 'category', 'manufacturer'])['manufacturer'].agg('nunique')

percent_per_user = unique_per_user.groupby('customer_id').apply(lambda x: sum(x)/total_man)

percent_per_user[percent_per_user  > 0.15].index.to_list()

[1001, 1003]

# 13
Какое максимальное количество продуктов было продано в один заказ?

In [57]:
# Группируем данные по order_id, затем суммируем по quantity и находим max
left_sales_pr = sales.merge(products, on='product_id', how='left')
left_sales_pr.groupby('order_id').quantity.sum().max()

np.int64(4)

# 14
Какой производитель имеет самую высокую среднюю цену продукта среди всех категорий?

In [ ]:
# Группируем данные по производителю, затем считаем средние цены за продукты всех категорий у каждого производителя и находим max
merged_data = sales.merge(products, on='product_id')

average_price_per_manufacturer_per_category = merged_data.groupby(['manufacturer', 'category'])['price_per_unit'].mean()

manufacturer_with_highest_average_price = average_price_per_manufacturer_per_category.groupby('manufacturer').mean().idxmax()

'E'

# 15
Каково общее количество уникальных производителей, чьи продукты были куплены менее чем в 3 регионах?

In [70]:
# Группируем по производителям и регионам и считаем в скольких у каждого производителя регионах были куплены товары и отбираем тех, у кого менее 3 регионов
merged_data = sales.merge(products, on='product_id')

merged_data.groupby(['manufacturer', 'region'])['region'].agg('nunique') \
    .groupby('manufacturer').apply(lambda x : len(x)) \
    .reset_index() \
    .query('region < 3').shape[0]

0